In [1]:
import os
from pathlib import Path

import pandas as pd
import numpy as np

In [2]:
ROOT_DIR = Path().cwd().parents[0]
os.chdir(ROOT_DIR)

from src.config import Data, Db_connection
os.chdir(ROOT_DIR/Data)
from load_tables import read_tables, DATA_DIR
os.chdir(ROOT_DIR/Db_connection)
from db_connection import engine

C:\Users\KISHORE\OneDrive\Pictures\Documents\internmo\-NexaTel-Customer-Churn-Analytics-Project
data dir C:\Users\KISHORE\OneDrive\Pictures\Documents\internmo\-NexaTel-Customer-Churn-Analytics-Project\data\raw


2026-09-29 16:02:29,560 - INFO - getting env variables...
2026-09-29 16:02:29,561 - INFO - Creadentials initiated Successfuly
2026-09-29 16:02:29,563 - INFO - DB URL Creating...
2026-09-29 16:02:29,564 - INFO - DB URL Initiated.
2026-09-29 16:02:29,565 - INFO - creating DB Engine...
2026-09-29 16:02:29,675 - INFO - engine created successfuly.


## **1) KPI: Customer Churn Rate**
### **KPI Purpose:** Target < 2.1% monthly. The headline retention metric.

### **KPI Definition:** Customers Lost / Customers at Start × 100

### **Data Sources:**
- customers Table
### **Exspected Output:** a single kpi value(Float) represeting churn rate.

In [3]:
customers_data = pd.read_sql_query(
    """
    SELECT *
    FROM customers;
    
    """,
    con = engine
)
customers_data.head(3)

,customer_id,first_name,last_name,gender,date_of_birth,age,email,phone_number,address,city_id,...,occupation,annual_income_inr,customer_segment,product_line,plan_id,acquisition_channel,acquisition_date,tenure_months,customer_status,churn_date
0,C0000001,Harish,Shetty,Male,1988-03-31,38,harish.shetty909@yahoo.in,916647481156,"Plot 382, Tilak Lane, Raipur",CT0026,...,School Teacher,791058.0,Mass,Prepaid Mobile,PL002,Retail Store,15-01-2026,2,Churned,22-03-2026
1,C0000002,Anjali,Khan,Female,1974-03-31,52,anjali.khan525@rediffmail.com,918116536619,"Door No. 269, Civil Lines Colony, Gurugram",CT0009,...,Small Business Owner,1385499.0,Premium,Fiber Broadband,PL017,Retail Store,01-01-2014,146,Active,None
2,C0000003,Arjun,Hegde,Male,1978-03-31,48,arjun.hegde378@hotmail.com,918958153134,"House No. 320, Garden Nagar, Vadodara",CT0020,...,Auto/Cab Driver,394773.0,Enterprise,Enterprise,PL021,Tele-sales,11-11-2015,1,Churned,22-12-2015


In [4]:
customers_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19000 entries, 0 to 18999
Data columns (total 25 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   customer_id          19000 non-null  object 
 1   first_name           19000 non-null  object 
 2   last_name            19000 non-null  object 
 3   gender               19000 non-null  object 
 4   date_of_birth        19000 non-null  object 
 5   age                  19000 non-null  int64  
 6   email                18256 non-null  object 
 7   phone_number         19000 non-null  object 
 8   address              18620 non-null  object 
 9   city_id              19000 non-null  object 
 10  city_name            19000 non-null  object 
 11  state_id             19000 non-null  object 
 12  pin_code             19000 non-null  object 
 13  city_tier            19000 non-null  object 
 14  region_id            19000 non-null  object 
 15  occupation           19000 non-null 

In [5]:
customers_data["acquisition_date"] = pd.to_datetime(customers_data["acquisition_date"], format="mixed")
customers_data["churn_date"] = pd.to_datetime(customers_data["churn_date"], format="mixed")

In [6]:
month_start = pd.Timestamp("2026-09-01")
month_end = pd.Timestamp("2023-10-31")

customers_at_start = customers_data[(customers_data["acquisition_date"] < month_start) & (customers_data["churn_date"].isna() | (customers_data["churn_date"] >= month_start))]["customer_id"].nunique()
customers_lost = customers_data[customers_data["churn_date"].between(month_start, month_end)]["customer_id"].nunique()

In [7]:
churn_rate = (customers_lost / customers_at_start) * 100
display(f"Customer Churn Rate: {churn_rate:.2f}%")

'Customer Churn Rate: 0.00%'

In [8]:
customers_data["acquisition_date"].max()

Timestamp('2026-12-02 00:00:00')

## **2) KPI: Customer Retention Rate**
### **KPI Purpose:** The positive mirror of churn.
### **KPI Definition:** (End - New) / Start × 100

### **Data Sources:**
- customers Table
### **Exspected Output:** a single kpi value(Float) represeting Retention Rate.

In [9]:
new_customers = customers_data[customers_data["acquisition_date"].between(month_start, month_end)]["customer_id"].nunique()
customers_at_end = customers_data[(customers_data["acquisition_date"] <= month_end) &(customers_data["churn_date"].isna() |(customers_data["churn_date"] > month_end))]["customer_id"].nunique()

In [10]:
retention_rate = ((customers_at_end - new_customers) / customers_at_start) * 100
display(f"Customer Retention Rate: {retention_rate:.2f}%")

'Customer Retention Rate: 92.87%'

## **3) KPI: ARPU**
### **KPI Purpose:** Monetisation per subscriber per month.
### **KPI Definition:** Total Recurring Revenue / Avg Active Subscribers

### **Data Sources:**
- subscriptions Table
### **Exspected Output:** a single kpi value(Float) represeting ARPU.

In [11]:
subscriptions_data = pd.read_sql_query(
    """
    SELECT *
    FROM subscriptions;
    
    """,
    con = engine
)
subscriptions_data.head(3)

,subscription_id,customer_id,plan_id,start_date,end_date,billing_cycle,monthly_charge_inr,subscription_status
0,S0000001,C0000001,PL002,2026-01-15,2026-03-22,Monthly,299.0,Terminated
1,S0000002,C0000002,PL017,2014-01-01,None,Monthly,1499.0,Active
2,S0000003,C0000003,PL021,2015-11-11,2015-12-22,Annual,4166.0,Terminated


In [12]:
active_subscriptions = subscriptions_data[subscriptions_data["subscription_status"] == "Active"]
active_subscribers = active_subscriptions["customer_id"].nunique()

total_recurring_revenue = active_subscriptions["monthly_charge_inr"].sum()
arpu = total_recurring_revenue / active_subscribers
display(f"ARPU: {arpu:,.2f}")

'ARPU: 951.92'

## **4) KPI: Customer Lifetime Value**
### **KPI Purpose:** Premium should be far higher than Mass.
### **KPI Definition:** ARPU × Gross Margin % × (1 / Churn Rate)

### **Data Sources:**
- customers Table
### **Exspected Output:** a single kpi value(Float) represeting CLV.

In [13]:
churned_customers = customers_data[customers_data["customer_status"] == "Churned"]
total_customers = len(customers_data)
churn_rate = (len(churned_customers) / total_customers) * 100

try:
    gross_margin_pct = 0.70
    churn_rate_decimal = churn_rate / 100

    clv = (arpu * gross_margin_pct * (1 / churn_rate_decimal))

    print(f"Customer Lifetime Value: ₹{clv:,.2f}")
except ZeroDivisionError as e:
    print("Error: ", e)

Customer Lifetime Value: ₹3,611.09


## **5) KPI: Revenue Lost to Churn**
### **KPI Purpose:** Translates churn into rupees at risk.
### **KPI Definition:** Churned Customers × their ARPU

### **Data Sources:**
- customers Table
### **Exspected Output:** a single kpi value(Float) represeting Revenue Lost to Churn .

In [14]:
churn_revenue = churned_customers.merge(subscriptions_data[["customer_id","monthly_charge_inr"]],on="customer_id",how="left")
revenue_lost_to_churn = (churn_revenue["monthly_charge_inr"].sum())
display(f"Revenue Lost to Churn: {revenue_lost_to_churn:,.2f}")

'Revenue Lost to Churn: 2,566,181.00'

## **6) KPI: Monthly Recurring Revenue**
### **KPI Purpose:** Size of the predictable revenue base.
### **KPI Definition:** Sum of active recurring charges

### **Data Sources:**
- subscriptions Table
### **Exspected Output:** a single kpi value(Float) represeting MRR.

In [15]:
mrr = active_subscriptions["monthly_charge_inr"].sum()
display(f"MRR : {mrr:,.2f}")

'MRR : 14,920,323.00'

## **7) KPI: First Contact Resolution**
### **KPI Purpose:** Target > 78%. Strong churn lever.
### **KPI Definition:** Resolved on First Contact / Total × 100

### **Data Sources:**
- support_tickets Table
### **Exspected Output:** a single kpi value(Float) represeting First Contact Resolution.

In [16]:
support_tickets_data = pd.read_sql_query(
    """
    SELECT *
    FROM support_tickets;
    
    """,
    con = engine
)
support_tickets_data.head(3)

,ticket_id,customer_id,created_date,category,priority,channel,status,first_contact_resolution,assigned_employee_id,resolution_hours,resolution_date
0,TK00000001,C0000001,08-09-2017,Billing Dispute,Medium,Store,Pending,No,EM00222,NaN,None
1,TK00000002,C0000001,12-01-2022,Slow Data Speed,Medium,Chat,Resolved,Yes,EM00466,25.6,13-01-2022
2,TK00000003,C0000002,21-09-2015,Network Issue,Low,Store,Pending,No,EM00462,NaN,None


In [17]:
fcr_rate = (support_tickets_data["first_contact_resolution"].eq("Yes").mean() * 100)

display(f"First Contact Resolution Rate : {fcr_rate:.2f}%")

'First Contact Resolution Rate : 40.41%'

## **8) KPI: Avg Resolution Time**
### **KPI Purpose:** Target < 24h. Long waits push customers out.
### **KPI Definition:** Mean of (resolved - created) hours

### **Data Sources:**
- support_tickets Table
### **Exspected Output:** a single kpi value(Float) represeting Avg Resolution Time.

In [18]:
avg_resolution_time = (support_tickets_data["resolution_hours"].mean())
display(f"Average Resolution Time : {avg_resolution_time:.2f} hours")

'Average Resolution Time : 16.04 hours'

## **9) KPI: Contract Renewal Rate**
### **KPI Purpose:** Target > 88% for postpaid / fibre.
### **KPI Definition:** Contracts Renewed / Due for Renewal × 100
### **Data Sources:**
- contracts Table
### **Exspected Output:** a single kpi value(Float) represeting Contract Renewal Rate.

In [19]:
contracts_data = pd.read_sql_query(
    """
    SELECT *
    FROM contracts;
    
    """,
    con = engine
)
contracts_data.head(3)

,contract_id,customer_id,plan_id,contract_type,contract_length_months,start_date,end_date,auto_renew,renewal_status
0,CN000001,C0000002,PL017,Fiber Broadband,24,01-01-2014,22-12-2015,Yes,Renewed
1,CN000002,C0000003,PL021,Enterprise,18,11-11-2015,04-05-2017,Yes,Not Renewed
2,CN000003,C0000004,PL006,Prepaid Mobile,12,18-10-2014,13-10-2015,No,Renewed


In [20]:
renewed_contracts = (contracts_data["renewal_status"].eq("Renewed").sum())
contracts_due = (contracts_data["renewal_status"].isin(["Renewed", "Not Renewed"]).sum())

renewal_rate = (renewed_contracts / contracts_due) * 100
display(f"Contract Renewal Rate: {renewal_rate:.2f}%")

'Contract Renewal Rate: 77.65%'

## **10) KPI: Avg Customer Tenure**
### **KPI Purpose:** Rising tenure = improving stickiness.
### **KPI Definition:** Mean tenure_months of active base
### **Data Sources:**
- customers Table
### **Exspected Output:** a single kpi value(Float) represeting Avg Customer Tenure.

In [21]:
active_customers = customers_data[customers_data["customer_status"] == "Active"]
avg_customer_tenure = (active_customers["tenure_months"].mean())

display(f"Average Customer Tenure: {avg_customer_tenure:.2f} months")

'Average Customer Tenure: 87.08 months'